In [0]:
from src.config import (
    CATALOG,
    SCHEMA,
    TRAIN_TABLE,
    TEST_TABLE,
    FEATURE_TABLE
)

print("CATALOG       :", CATALOG)
print("SCHEMA        :", SCHEMA)
print("TRAIN TABLE   :", TRAIN_TABLE)
print("TEST TABLE    :", TEST_TABLE)
print("FEATURE TABLE :", FEATURE_TABLE)

In [0]:
from src.feature_engineering import (
    create_features,
    validate_features,
    get_feature_columns
)

print("Feature engineering functions imported.")

In [0]:
train_df = spark.table(TRAIN_TABLE)

print("Training rows:", train_df.count())

display(train_df)

In [0]:
train_df.printSchema()

In [0]:
from src.feature_engineering import (
    validate_input_columns,
    validate_key
)

validate_input_columns(train_df)

validate_key(train_df)

In [0]:
feature_df = create_features(train_df)

display(feature_df)

In [0]:
display(
    feature_df.select(
        "record_id",
        "sepal_length",
        "sepal_width",
        "petal_length",
        "petal_width",
        "petal_to_sepal_length_ratio",
        "petal_to_sepal_width_ratio",
        "target",
        "species"
    )
)

In [0]:
validate_features(feature_df)

In [0]:
feature_columns = get_feature_columns()

print("Model feature columns:")
for column in feature_columns:
    print(" -", column)

In [0]:
display(
    feature_df.select(feature_columns).describe()
)

In [0]:
for column in feature_columns:
    correlation = (
        feature_df.stat.corr(
            column,
            "target"
        )
    )

    print(
        f"{column:40s} "
        f"correlation = {correlation:.4f}"
    )

In [0]:
feature_table_df = feature_df.select(
    "record_id",
    *feature_columns,
    "target",
    "species"
)

display(feature_table_df)

In [0]:
feature_count = feature_table_df.count()

print("Feature table rows:", feature_count)

assert feature_count == train_df.count(), (
    "Feature row count does not match training row count."
)

print("Feature row count validation: PASSED")

In [0]:
duplicate_keys = (
    feature_table_df
    .groupBy("record_id")
    .count()
    .filter("count > 1")
)

duplicate_key_count = duplicate_keys.count()

print(
    "Duplicate record_id values:",
    duplicate_key_count
)

assert duplicate_key_count == 0, (
    "record_id must be unique."
)

print("Primary key validation: PASSED")

In [0]:
(
    feature_table_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(FEATURE_TABLE)
)

print(
    f"Feature table created: {FEATURE_TABLE}"
)

In [0]:
stored_feature_df = spark.table(FEATURE_TABLE)

print(
    "Stored feature rows:",
    stored_feature_df.count()
)

display(stored_feature_df)

In [0]:
stored_count = stored_feature_df.count()

assert stored_count == feature_count, (
    f"Stored feature count {stored_count} "
    f"does not match expected {feature_count}"
)

print("Persisted feature table validation: PASSED")

In [0]:
validate_features(stored_feature_df)

print()
print("==========================================")
print("FEATURE ENGINEERING COMPLETED")
print("==========================================")
print(f"Feature table : {FEATURE_TABLE}")
print(f"Rows          : {stored_count}")
print(f"Features      : {len(feature_columns)}")
print("==========================================")